# Observing Arabic Dialects with NLP Libraries and Open-Source Repositories

## Learning Goal

By the end of this notebook, you will understand how to build a small **Arabic dialect observation workflow**.

This is not only about predicting one dialect label. It is about learning how to observe, compare, debug, and document dialect signals in Arabic text.

You will learn:

- What Arabic dialect identification means
- How dialects differ from Modern Standard Arabic
- How to start with simple rules before using models
- How to use libraries such as CAMeL Tools and Hugging Face models
- How open-source GitHub repositories and datasets fit the workflow
- How dialect observation connects to RAG, APIs, ML pipelines, ingestion, evaluation, and vector databases

### Background & Links

Useful libraries, models, datasets, and repositories:

- CAMeL Tools GitHub: https://github.com/CAMeL-Lab/camel_tools
- CAMeL Tools dialect identification docs: https://camel-tools.readthedocs.io/en/latest/api/dialectid.html
- CAMeLBERT GitHub: https://github.com/CAMeL-Lab/CAMeLBERT
- CAMeLBERT-Mix DID MADAR 26 model: https://huggingface.co/CAMeL-Lab/bert-base-arabic-camelbert-mix-did-madar-corpus26
- CAMeLBERT dialectal Arabic model: https://huggingface.co/CAMeL-Lab/bert-base-arabic-camelbert-da
- ARBERT and MARBERT collection: https://huggingface.co/collections/UBC-NLP/arbert-and-marbert
- Arabic MARBERT dialect identification city model: https://huggingface.co/Ammar-alhaj-ali/arabic-MARBERT-dialect-identification-city
- MADAR shared task paper: https://aclanthology.org/W19-4622/
- Arabic dialect identification GitHub example: https://github.com/swshon/arabic-dialect-identification
- Arabic dialect identification experiments: https://github.com/amurtadha/arabic-dialect-identification
- ADI under scrutiny repo: https://github.com/AMR-KELEG/ADI-under-scrutiny
- CODAfication repo: https://github.com/CAMeL-Lab/codafication

### Why it matters in real AI engineering

Arabic dialects affect many AI systems:

- search quality
- chatbot understanding
- classification
- moderation
- sentiment analysis
- customer support routing
- speech and transcription pipelines
- RAG retrieval over social or conversational text
- translation and dialect-to-MSA normalization

A model trained only on Modern Standard Arabic may fail on dialectal Arabic.

Example:

```text
MSA: ماذا تريد أن تفعل؟
Levantine: شو بدك تعمل؟
Egyptian: عايز تعمل إيه؟
Gulf: وش تبي تسوي؟
Maghrebi: شنو بغيتي دير؟
```

## Mental Model

Think of dialect observation as a layered process.

```text
Raw Arabic text
    ↓
cleaning and normalization
    ↓
dialect signal observation
    ↓
rule-based baseline
    ↓
model-based prediction
    ↓
error analysis
    ↓
production decision
```

### What are we observing?

Dialect can appear through:

- vocabulary
- question words
- negation patterns
- pronouns
- particles
- spelling habits
- code-switching
- named entities and locations
- morphology
- phonological spelling variation

### How this concept fits production systems

In production, dialect detection can be used as a routing layer.

```text
User message
    ↓
dialect detector
    ↓
route to best model / prompt / retriever / normalization pipeline
    ↓
answer or classify
```

Dialect detection is not always the final product. Often, it is an internal system component that improves downstream behavior.

## Background Explanation

### Core concepts

#### 1. Arabic dialect identification

Arabic dialect identification is the task of predicting the dialect or dialect region of a given Arabic text.

Possible label types:

- broad region: Levantine, Egyptian, Gulf, Maghrebi, MSA
- country: Syria, Egypt, Saudi Arabia, Morocco
- city: Cairo, Damascus, Beirut, Riyadh, Casablanca
- binary: MSA vs dialectal
- mixed: MSA + dialect + English

#### 2. Dialect observation vs dialect classification

Classification asks:

```text
What is the dialect label?
```

Observation asks:

```text
What signals suggest this dialect?
Which words or patterns caused this guess?
Where is the model uncertain?
```

Observation is better for learning and debugging.

#### 3. Rule-based baseline

A rule-based baseline uses known dialect markers.

Example:

```text
شو، بدك، هلق → Levantine signals
عايز، إيه، أوي → Egyptian signals
وش، تبي، وايد → Gulf signals
بغيت، شنو، بزاف → Maghrebi signals
```

Rules are not enough for production, but they are excellent for understanding.

#### 4. Model-based dialect identification

Libraries and models can predict dialect labels.

Examples:

- CAMeL Tools dialect identification component
- CAMeLBERT DID models
- MARBERT-based dialect identification models
- models fine-tuned on MADAR, NADI, QADI, or ADI-style datasets

#### 5. Dataset awareness

Dialect datasets differ by:

- domain: travel, Twitter, broadcast speech, comments
- granularity: city, country, region
- text length
- label quality
- dialect coverage
- licensing

Never assume one dataset represents all Arabic dialect usage.

#### 6. Error analysis

Dialect models often fail on:

- short text
- mixed dialects
- MSA-like dialect sentences
- code-switching
- named entities
- spelling variation
- sarcastic or informal posts
- dialects underrepresented in training data

## Setup

This notebook uses two levels:

### Level 1 — Always runnable

Uses:

- Python
- regex
- pandas
- simple rule-based dialect markers

### Level 2 — Optional model-based usage

Uses:

- CAMeL Tools
- Hugging Face Transformers
- CAMeLBERT or MARBERT dialect models

Some model-based components may require:

- internet access
- model downloads
- Linux/macOS for some CAMeL Tools dialect ID components
- enough memory

The exercises are designed so you can still learn even if the model download does not work.

In [18]:
# Install basic packages if needed.
# Optional heavy packages are not installed automatically here to keep the notebook lightweight.

import sys
import subprocess
import importlib.util

basic_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
}

missing = [pip_name for module_name, pip_name in basic_packages.items() if importlib.util.find_spec(module_name) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Basic packages are already installed.")

Basic packages are already installed.


In [19]:
import re
import pandas as pd
import numpy as np
from collections import Counter, defaultdict

print("Imports completed.")

Imports completed.


## Minimal Working Example

We will build a tiny dialect observation tool.

It will:

1. normalize Arabic text lightly
2. search for dialect markers
3. count signals by dialect region
4. return a simple observation report

This is not a production dialect classifier. It is a learning baseline.

In [4]:
def light_normalize_arabic(text: str) -> str:
    text = re.sub(r"ـ", "", text)
    text = re.sub(r"[ًٌٍَُِّْ]", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

dialect_markers = {
    "levantine": ["شو", "بدك", "بدي", "هلق", "كتير", "لسا", "ليش"],
    "egyptian": ["ايه", "عايز", "عاوزه", "اوي", "ازاي", "دلوقتي", "مش"],
    "gulf": ["وش", "تبي", "ابي", "وايد", "الحين", "ليش", "مو"],
    "maghrebi": ["شنو", "بغيت", "بزاف", "دابا", "واش", "علاش"],
    "msa": ["ماذا", "اريد", "الان", "كثيرا", "لماذا", "ليس"],
}

def observe_dialect_signals(text: str, markers: dict) -> dict:
    normalized = light_normalize_arabic(text)
    scores = {}
    matched = {}

    for dialect, words in markers.items():
        hits = [word for word in words if word in normalized]
        scores[dialect] = len(hits)
        matched[dialect] = hits

    best_dialect = max(scores, key=scores.get)
    print(f'best_dialect: {best_dialect}')
    print(scores)
    print(matched)

    return {
        "raw_text": text,
        "normalized_text": normalized,
        "best_guess": best_dialect if scores[best_dialect] > 0 else "unknown",
        "scores": scores,
        "matched_markers": matched,
    }

examples = [
    "شو بدك تعمل هلق؟",
    "عايز تروح فين دلوقتي؟",
    "وش تبي تسوي الحين؟",
    "شنو بغيتي دير دابا؟",
    "ماذا تريد أن تفعل الآن؟",
]

for text in examples:
    report = observe_dialect_signals(text, dialect_markers)
    print("Text:", report["raw_text"])
    print("Best guess:", report["best_guess"])
    print("Matched:", report["matched_markers"])
    print("-" * 80)

best_dialect: levantine
{'levantine': 3, 'egyptian': 0, 'gulf': 0, 'maghrebi': 0, 'msa': 0}
{'levantine': ['شو', 'بدك', 'هلق'], 'egyptian': [], 'gulf': [], 'maghrebi': [], 'msa': []}
Text: شو بدك تعمل هلق؟
Best guess: levantine
Matched: {'levantine': ['شو', 'بدك', 'هلق'], 'egyptian': [], 'gulf': [], 'maghrebi': [], 'msa': []}
--------------------------------------------------------------------------------
best_dialect: egyptian
{'levantine': 0, 'egyptian': 2, 'gulf': 0, 'maghrebi': 0, 'msa': 0}
{'levantine': [], 'egyptian': ['عايز', 'دلوقتي'], 'gulf': [], 'maghrebi': [], 'msa': []}
Text: عايز تروح فين دلوقتي؟
Best guess: egyptian
Matched: {'levantine': [], 'egyptian': ['عايز', 'دلوقتي'], 'gulf': [], 'maghrebi': [], 'msa': []}
--------------------------------------------------------------------------------
best_dialect: gulf
{'levantine': 0, 'egyptian': 0, 'gulf': 3, 'maghrebi': 0, 'msa': 0}
{'levantine': [], 'egyptian': [], 'gulf': ['وش', 'تبي', 'الحين'], 'maghrebi': [], 'msa': []}
Tex

In [22]:
# print(type(dialect_markers.items()))
# print(type(dialect_markers))

# for x in dialect_markers:
#     print(x)

# for x in dialect_markers.items():
#     print(x)

# for x in dialect_markers.keys():
#     print(x)

# for x in dialect_markers.values():
#     print(x)

scores = {'levantine': 3, 'egyptian': 0, 'gulf': 0, 'maghrebi': 0, 'msa': 0}
# best_dialect = max(scores, )
# best_dialect
scores.get

<function dict.get(key, default=None, /)>

### Short explanation

This simple baseline works by counting visible dialect markers.

It teaches an important engineering idea:

> Before trusting a model, build a transparent baseline.

A transparent baseline helps you:

- understand the task
- inspect errors
- explain predictions
- build test cases
- compare model behavior

## Exercise 1

### Goal

Improve the rule-based dialect observer.

You will:

1. add more dialect markers
2. create a function that returns the top 2 dialect guesses
3. return matched words for each dialect
4. handle texts with no matches

### TODO section

Complete the code below.

### Questions to think about

- Which markers are strong dialect signals?
- Which markers are ambiguous?
- Why is `ليش` not enough to distinguish Levantine and Gulf?
- Why should the tool return evidence, not only a label?

In [44]:
from operator import itemgetter
# Exercise 1 TODO

custom_markers = {
    "levantine": ["شو", "بدك", "بدي", "هلق"],
    "egyptian": ["ايه", "عايز", "اوي", "ازاي"],
    "gulf": ["وش", "تبي", "وايد", "الحين"],
    "maghrebi": ["شنو", "بغيت", "بزاف", "دابا"],
    "msa": ["ماذا", "اريد", "الان", "كثيرا"],
}

def top_dialect_guesses(text: str, markers: dict, top_n: int = 2) -> dict:

    hits = {}

    # TODO 1: normalize the text
    light_normalize_arabic(text=text)

    # TODO 2: count marker hits per dialect
    for dialect, words in markers.items():
        for word in words:
            if word in text:
                hits[dialect] = hits.get(dialect, 0) + 1

    print(hits)
    # TODO 3: sort dialects by score descending
    # sorted = dict(sorted(hits.items(), key=lambda item: item[1], reverse=True))

    # print(sorted)
    # TODO 4: return raw_text, normalized_text, top_guesses, matched_markers, status

sample = "بدي اعرف شو بغيت وايد تبي اوي ايه كيف الحين هلق"
top_dialect_guesses(sample, custom_markers)

{'levantine': 3, 'egyptian': 2, 'gulf': 3, 'maghrebi': 1}


## Solution 1

In [20]:
# Solution 1

custom_markers = {
    "levantine": ["شو", "بدك", "بدي", "هلق", "كتير", "لسا", "وين"],
    "egyptian": ["ايه", "عايز", "عاوزه", "اوي", "ازاي", "دلوقتي", "فين"],
    "gulf": ["وش", "تبي", "ابي", "وايد", "الحين", "وين", "مو"],
    "maghrebi": ["شنو", "بغيت", "بزاف", "دابا", "واش", "فين", "علاش"],
    "msa": ["ماذا", "اريد", "الان", "كثيرا", "اين", "لماذا", "ليس"],
}

def top_dialect_guesses(text: str, markers: dict, top_n: int = 2) -> dict:
    normalized = light_normalize_arabic(text)
    scores = {}
    matched_markers = {}

    for dialect, words in markers.items():
        hits = [word for word in words if word in normalized]
        scores[dialect] = len(hits)
        matched_markers[dialect] = hits

    sorted_scores = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    top_guesses = sorted_scores[:top_n]
    max_score = top_guesses[0][1] if top_guesses else 0
    status = "matched" if max_score > 0 else "no_signal"

    return {
        "raw_text": text,
        "normalized_text": normalized,
        "top_guesses": top_guesses,
        "matched_markers": matched_markers,
        "status": status,
    }

samples = [
    "بدي اعرف شو صار هلق",
    "عايز اعرف حصل ايه دلوقتي",
    "وش تبي تسوي الحين",
    "هذا نص بدون مؤشرات واضحة",
]

for sample in samples:
    result = top_dialect_guesses(sample, custom_markers)
    print(result)
    print("-" * 80)

{'raw_text': 'بدي اعرف شو صار هلق', 'normalized_text': 'بدي اعرف شو صار هلق', 'top_guesses': [('levantine', 3), ('egyptian', 0)], 'matched_markers': {'levantine': ['شو', 'بدي', 'هلق'], 'egyptian': [], 'gulf': [], 'maghrebi': [], 'msa': []}, 'status': 'matched'}
--------------------------------------------------------------------------------
{'raw_text': 'عايز اعرف حصل ايه دلوقتي', 'normalized_text': 'عايز اعرف حصل ايه دلوقتي', 'top_guesses': [('egyptian', 3), ('levantine', 0)], 'matched_markers': {'levantine': [], 'egyptian': ['ايه', 'عايز', 'دلوقتي'], 'gulf': [], 'maghrebi': [], 'msa': []}, 'status': 'matched'}
--------------------------------------------------------------------------------
{'raw_text': 'وش تبي تسوي الحين', 'normalized_text': 'وش تبي تسوي الحين', 'top_guesses': [('gulf', 3), ('levantine', 0)], 'matched_markers': {'levantine': [], 'egyptian': [], 'gulf': ['وش', 'تبي', 'الحين'], 'maghrebi': [], 'msa': []}, 'status': 'matched'}
-------------------------------------------

## Exercise 2

### Slightly harder

Build a small dialect observation dataframe.

You will:

1. create a small dataset of Arabic sentences
2. run the observer on each sentence
3. store predictions, scores, and evidence
4. compare prediction against a weak gold label
5. inspect errors

### Questions to think about

- Why is a tiny evaluation set useful before training a model?
- What kinds of examples should be in the test set?
- How do you distinguish model failure from bad labeling?
- Why should errors be inspected manually?

In [38]:
# Exercise 2 TODO

toy_dataset = [
    {"text": "شو بدك تعمل هلق؟", "gold": "levantine"},
    {"text": "عايز تروح فين دلوقتي؟", "gold": "egyptian"},
    {"text": "وش تبي تسوي الحين؟", "gold": "gulf"},
    {"text": "شنو بغيتي دير دابا؟", "gold": "maghrebi"},
    {"text": "ماذا تريد أن تفعل الآن؟", "gold": "msa"},
    {"text": "ليش ما رحت؟", "gold": "ambiguous"},
]

# TODO:
# 1. Run top_dialect_guesses on every row.

# 2. Store predicted dialect.

# 3. Store score.
# 4. Store matched evidence.
# 5. Create a pandas DataFrame.
# 6. Add a column is_correct.
# 7. Show incorrect rows.
gold_datasets = []
for toy_example in toy_dataset:
    results = top_dialect_guesses(toy_example['text'], custom_markers)
    print(results)
    # print(results['top_guesses'])
    # print(type(results['top_guesses']))
    # gold_datasets["predicted dialect"] = results['top_guesses']
    gold_datasets.append({
      "predicted dialect": results['top_guesses'][0][0],
      "Score": results['top_guesses'][0][1]
      
    })

    
df = pd.DataFrame(gold_datasets)
df

# Each dictionary:
# keys → columns
# values → cell values
# Python list of dicts
#         ↓
# Pandas converts it
#         ↓
# Spreadsheet-like table

{'raw_text': 'شو بدك تعمل هلق؟', 'normalized_text': 'شو بدك تعمل هلق؟', 'top_guesses': [('levantine', 3), ('egyptian', 0)], 'matched_markers': {'levantine': ['شو', 'بدك', 'هلق'], 'egyptian': [], 'gulf': [], 'maghrebi': [], 'msa': []}, 'status': 'matched'}
{'raw_text': 'عايز تروح فين دلوقتي؟', 'normalized_text': 'عايز تروح فين دلوقتي؟', 'top_guesses': [('egyptian', 3), ('maghrebi', 1)], 'matched_markers': {'levantine': [], 'egyptian': ['عايز', 'دلوقتي', 'فين'], 'gulf': [], 'maghrebi': ['فين'], 'msa': []}, 'status': 'matched'}
{'raw_text': 'وش تبي تسوي الحين؟', 'normalized_text': 'وش تبي تسوي الحين؟', 'top_guesses': [('gulf', 3), ('levantine', 0)], 'matched_markers': {'levantine': [], 'egyptian': [], 'gulf': ['وش', 'تبي', 'الحين'], 'maghrebi': [], 'msa': []}, 'status': 'matched'}
{'raw_text': 'شنو بغيتي دير دابا؟', 'normalized_text': 'شنو بغيتي دير دابا؟', 'top_guesses': [('maghrebi', 3), ('levantine', 0)], 'matched_markers': {'levantine': [], 'egyptian': [], 'gulf': [], 'maghrebi': ['شن

,predicted dialect,Score
0,levantine,3
1,egyptian,3
2,gulf,3
3,maghrebi,3
4,msa,2
5,levantine,0


## Solution 2

In [30]:
# Solution 2

toy_dataset = [
    {"text": "شو بدك تعمل هلق؟", "gold": "levantine"},
    {"text": "عايز تروح فين دلوقتي؟", "gold": "egyptian"},
    {"text": "وش تبي تسوي الحين؟", "gold": "gulf"},
    {"text": "شنو بغيتي دير دابا؟", "gold": "maghrebi"},
    {"text": "ماذا تريد أن تفعل الآن؟", "gold": "msa"},
    {"text": "ليش ما رحت؟", "gold": "ambiguous"},
]

rows = []

for item in toy_dataset:
    result = top_dialect_guesses(item["text"], custom_markers, top_n=2)
    top_guess, top_score = result["top_guesses"][0]

    rows.append(
        {
            "text": item["text"],
            "gold": item["gold"],
            "prediction": top_guess if result["status"] == "matched" else "unknown",
            "score": top_score,
            "top_guesses": result["top_guesses"],
            "evidence": result["matched_markers"],
            "status": result["status"],
        }
    )

print(type(rows))
print(type(rows[0]))
df = pd.DataFrame(rows)
df["is_correct"] = df["gold"] == df["prediction"]

display(df)

print("Accuracy on non-ambiguous labels:")
non_ambiguous = df[df["gold"] != "ambiguous"]
print(non_ambiguous["is_correct"].mean())

print("\nPotential errors or ambiguous cases:")
display(df[(df["is_correct"] == False) | (df["gold"] == "ambiguous")])

<class 'list'>
<class 'dict'>


,text,gold,prediction,score,top_guesses,evidence,status,is_correct
0,شو بدك تعمل هلق؟,levantine,levantine,3,"[(levantine, 3), (egyptian, 0)]","{'levantine': ['شو', 'بدك', 'هلق'], 'egyptian'...",matched,True
1,عايز تروح فين دلوقتي؟,egyptian,egyptian,3,"[(egyptian, 3), (maghrebi, 1)]","{'levantine': [], 'egyptian': ['عايز', 'دلوقتي...",matched,True
2,وش تبي تسوي الحين؟,gulf,gulf,3,"[(gulf, 3), (levantine, 0)]","{'levantine': [], 'egyptian': [], 'gulf': ['وش...",matched,True
3,شنو بغيتي دير دابا؟,maghrebi,maghrebi,3,"[(maghrebi, 3), (levantine, 0)]","{'levantine': [], 'egyptian': [], 'gulf': [], ...",matched,True
4,ماذا تريد أن تفعل الآن؟,msa,msa,2,"[(msa, 2), (levantine, 0)]","{'levantine': [], 'egyptian': [], 'gulf': [], ...",matched,True
5,ليش ما رحت؟,ambiguous,unknown,0,"[(levantine, 0), (egyptian, 0)]","{'levantine': [], 'egyptian': [], 'gulf': [], ...",no_signal,False


Accuracy on non-ambiguous labels:
1.0

Potential errors or ambiguous cases:


,text,gold,prediction,score,top_guesses,evidence,status,is_correct
5,ليش ما رحت؟,ambiguous,unknown,0,"[(levantine, 0), (egyptian, 0)]","{'levantine': [], 'egyptian': [], 'gulf': [], ...",no_signal,False


## Optional Model-Based Example

This optional section shows how you might call an open-source Hugging Face dialect model.

Use this only if you have internet access and enough memory.

Example model candidates:

- `CAMeL-Lab/bert-base-arabic-camelbert-mix-did-madar-corpus26`
- `Ammar-alhaj-ali/arabic-MARBERT-dialect-identification-city`

The exact labels depend on the model and dataset.

In [ ]:
# Optional: Hugging Face model-based dialect identification
# This may download a model and can fail if offline.

def try_huggingface_dialect_model(texts):
    try:
        from transformers import pipeline

        model_name = "CAMeL-Lab/bert-base-arabic-camelbert-mix-did-madar-corpus26"
        classifier = pipeline("text-classification", model=model_name, tokenizer=model_name, top_k=3)

        return classifier(texts)

    except Exception as e:
        print("Could not run Hugging Face dialect model.")
        print("Reason:", type(e).__name__, str(e)[:300])
        return None

# Uncomment to try:
# try_huggingface_dialect_model(["شو بدك تعمل هلق؟", "عايز تروح فين دلوقتي؟"])

## Common Mistakes

### Mistake 1: Assuming one word proves a dialect

Some words appear in multiple dialects.

Example:

```text
ليش
وين
مو
```

These can occur across different regions.

A better approach is to combine many signals.

### Mistake 2: Ignoring text length

Short texts are hard.

```text
تمام
ايوا
لا
ليش
```

These may not contain enough evidence.

### Mistake 3: Treating dialect labels as objective truth

Dialect labels can be fuzzy.

A speaker may mix:

- MSA
- local dialect
- another regional dialect
- English or French
- internet slang

### Mistake 4: Training without checking dataset domain

A model trained on travel sentences may not work well on Twitter, YouTube, or religious discussions.

### Mistake 5: Confusing country, city, and regional labels

These are different tasks.

```text
Region-level: Levantine
Country-level: Syria
City-level: Damascus
```

A model trained for city labels should not be blindly evaluated as a regional model.

### Mistake 6: Removing too much information during normalization

Over-normalization can erase useful dialect clues.

Best practice:

```text
store raw_text
store normalized_text
store detected markers
store model prediction
```

### Mistake 7: Forgetting licensing and data access

Many Arabic dialect datasets have specific licenses or access rules.

Always check dataset terms before building a public demo or commercial system.

In [ ]:
# Debugging example: ambiguous marker

ambiguous_texts = [
    "ليش ما جيت؟",
    "وين رايح؟",
    "مو مشكلة",
]

for text in ambiguous_texts:
    result = top_dialect_guesses(text, custom_markers, top_n=3)
    print("Text:", text)
    print("Top guesses:", result["top_guesses"])
    print("Evidence:", result["matched_markers"])
    print("Observation: this is probably not enough evidence for a confident label.")
    print("-" * 80)

## Real AI Engineering Usage

### 1. RAG

Dialect observation can improve RAG over informal Arabic.

Example:

```text
User asks in Levantine
    ↓
detect dialect
    ↓
normalize or expand query
    ↓
retrieve dialect-aware chunks
    ↓
generate answer in matching style
```

For Arabic RAG, dialect detection can help with:

- query rewriting
- retriever routing
- answer style control
- dialect-to-MSA normalization
- detecting mismatches between query and corpus

### 2. APIs

A FastAPI service might expose:

```text
POST /dialect/observe
POST /dialect/classify
POST /dialect/normalize
POST /search
```

Example response:

```json
{
  "text": "شو بدك تعمل هلق؟",
  "prediction": "levantine",
  "confidence": 0.82,
  "evidence": ["شو", "بدك", "هلق"]
}
```

### 3. ML pipelines

Dialect observation can be part of:

- dataset profiling
- labeling support
- weak supervision
- feature engineering
- active learning
- error analysis
- data balancing

### 4. Ingestion

During ingestion, dialect metadata can be attached to documents or chunks.

Example metadata:

```python
{
    "chunk_id": "comment-001",
    "dialect_prediction": "egyptian",
    "dialect_confidence": 0.77,
    "dialect_evidence": ["عايز", "ايه"],
    "source": "comments.csv"
}
```

### 5. Evaluation

Dialect-aware evaluation can answer:

- Does the model fail more on Maghrebi text?
- Does retrieval work better for MSA than dialect?
- Are labels too coarse?
- Are short texts being overclassified?
- Are ambiguous samples marked as uncertain?

### 6. Vector DBs

A vector database can store dialect metadata in payload.

Example Qdrant payload:

```python
{
    "text": "شو بدك تعمل هلق؟",
    "dialect": "levantine",
    "confidence": 0.82,
    "source": "social_comments",
    "language_variant": "dialectal_arabic"
}
```

Then you can filter retrieval:

```text
search only dialect = levantine
search only MSA
search mixed dialect + MSA
```

## Final Mini Exercise

### Small production-style task

Build a dialect observation pipeline function:

```python
observe_dataset_dialects(records)
```

Each record should include:

```python
{
    "id": "...",
    "text": "...",
    "source": "..."
}
```

The function should return a DataFrame with:

- id
- source
- raw_text
- normalized_text
- predicted_dialect
- score
- evidence
- status
- needs_human_review

Rules:

- `needs_human_review = True` if:
  - status is `no_signal`
  - top score is 0
  - top two dialects have the same score

In [ ]:
# Final Mini Exercise TODO

records = [
    {"id": "msg-001", "text": "شو بدك تعمل هلق؟", "source": "chat"},
    {"id": "msg-002", "text": "عايز افهم الموضوع ده", "source": "chat"},
    {"id": "msg-003", "text": "ليش ما رحت؟", "source": "comments"},
    {"id": "msg-004", "text": "هذا نص عربي فصيح", "source": "article"},
]

def observe_dataset_dialects(records):
    # TODO:
    # 1. Loop over records
    # 2. Run top_dialect_guesses
    # 3. Extract top dialect and score
    # 4. Detect ties between top two guesses
    # 5. Set needs_human_review
    # 6. Return pandas DataFrame
    pass

# observe_dataset_dialects(records)

### Final Mini Exercise Solution

In [ ]:
# Final Mini Exercise Solution

records = [
    {"id": "msg-001", "text": "شو بدك تعمل هلق؟", "source": "chat"},
    {"id": "msg-002", "text": "عايز افهم الموضوع ده", "source": "chat"},
    {"id": "msg-003", "text": "ليش ما رحت؟", "source": "comments"},
    {"id": "msg-004", "text": "هذا نص عربي فصيح", "source": "article"},
]

def observe_dataset_dialects(records):
    rows = []

    for record in records:
        result = top_dialect_guesses(record["text"], custom_markers, top_n=2)

        top_guess, top_score = result["top_guesses"][0]
        second_guess, second_score = result["top_guesses"][1]

        has_tie = top_score == second_score
        no_signal = result["status"] == "no_signal" or top_score == 0

        needs_human_review = bool(no_signal or has_tie)

        rows.append(
            {
                "id": record["id"],
                "source": record["source"],
                "raw_text": record["text"],
                "normalized_text": result["normalized_text"],
                "predicted_dialect": top_guess if not no_signal else "unknown",
                "score": top_score,
                "evidence": result["matched_markers"],
                "top_guesses": result["top_guesses"],
                "status": result["status"],
                "needs_human_review": needs_human_review,
            }
        )

    return pd.DataFrame(rows)

observed_df = observe_dataset_dialects(records)
display(observed_df)

## Key Takeaways

- Arabic dialect work should start with observation, not blind modeling.
- Dialect signals include vocabulary, particles, pronouns, negation, spelling, and context.
- Rule-based baselines are weak but very useful for learning and debugging.
- CAMeL Tools, CAMeLBERT, MARBERT, MADAR, NADI, QADI, and ADI-style repositories are useful starting points.
- Short Arabic texts are often ambiguous.
- Dialect labels can be regional, country-level, city-level, or mixed.
- Always store evidence, not only predictions.
- Store raw text and normalized text separately.
- In production, dialect detection can route queries, improve retrieval, support normalization, and improve evaluation.
- For vector databases, dialect predictions can be stored as metadata and used as retrieval filters.